In [1]:
from langchain_milvus_ingest import store_chunks_to_milvus

C:\Users\USER\Miniconda3\envs\kpn-ai\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from chunking import pages_to_chunks
from pdf_with_tables import extract_pdf_content

In [3]:
import os

In [9]:
from pymilvus import connections, utility, Collection

In [4]:
folder_path = 'assets'

In [5]:
# for filename in os.listdir(folder_path):
#     filepath = os.path.join(folder_path, filename)
#     print(filepath)
#     pdf_pages = extract_pdf_content(filepath)
#     filename = os.path.basename(filepath)
#     all_chunks = pages_to_chunks(pdf_pages, source=filename, language='id')
    

In [6]:
all_chunks = []
for filename in os.listdir(folder_path):
    filepath = os.path.join(folder_path, filename)
    if filepath.endswith('.pdf'):
        print(filepath)
        pdf_pages = extract_pdf_content(filepath)
        
        filename = os.path.basename(filepath)
        chunks = pages_to_chunks(pdf_pages, source=filename)
        all_chunks.extend(chunks)

assets\CP-Bantuan-Pengobatan-2025.pdf
assets\CP-Cuti-dan-Izin-Khusus.pdf
assets\CP_Perjalanan Dinas.pdf


In [7]:
all_chunks[-50]

{'chunk_id': '3fc075643a5f4a3f9ce2f0b2c43fb24a',
 'source': 'CP_Perjalanan Dinas.pdf',
 'page': 18,
 'element_type': 'narrative',
 'table_id': None,
 'row_id': None,
 'bbox': None,
 'language': 'id',
 'chunk_text': 'PERJALANAN DINAS – TRANSPORTASI PERJALANAN DINAS',
 'checksum': 'bb975e35772928f3d82b03e035f0a77a0eb2fd2625f73e6ab6b9a5283f3cfdc1',
 'created_at': '2025-09-23T10:31:00Z'}

In [8]:
# if utility.has_collection("company_policies"):
#     utility.drop_collection("company_policies")
#     print("deleted")

NameError: name 'utility' is not defined

In [10]:
skipped, inserted = store_chunks_to_milvus(
    chunks=all_chunks,
    collection_name="company_policies",
    milvus_host="127.0.0.1",
    milvus_port="19530",
    embedding_model="models/embedding-001",
    embed_batch_size=32,
    dedupe_with_milvus=True,
    dry_run=False
)
print("skipped, inserted =", skipped, inserted)


INFO:langchain_milvus_ingest:Created Milvus collection 'company_policies' with dim=768
INFO:langchain_milvus_ingest:Finished ingest: inserted=180, skipped=0


skipped, inserted = 0 180


In [11]:
HOST = "127.0.0.1"
PORT = "19530"
COL = "company_policies"  # your collection name

# 1) connect
connections.connect(alias="default", host=HOST, port=PORT)

# 2) show collections
print("Collections:", utility.list_collections())

# 3) does collection exist?
print("Has collection:", utility.has_collection(COL))

# 4) inspect schema (if exists)
if utility.has_collection(COL):
    c = Collection(COL)
    print("Num entities:", c.num_entities)
    print("Schema fields:")
    for f in c.schema.fields:
        print("  ", f.name, f.dtype, getattr(f, "dim", None))

# 5) run a checksum query (example)
if utility.has_collection(COL):
    rows = c.query(expr='checksum in ["sha256-some"]', output_fields=["checksum","metadata"])
    print(rows)


Collections: ['pdf_docs_test', 'milvus_langchain_test', 'company_policies']
Has collection: True
Num entities: 180
Schema fields:
   id 21 None
   embedding 101 768
   checksum 21 None
   metadata 23 None
data: [], extra_info: {}


In [12]:
coll = c

In [ ]:
# assume coll exists from previous snippet
checksum = "sha256-your-checksum-here"
rows = coll.query(expr=f'checksum in ["{checksum}"]', output_fields=["id","checksum","metadata"])
print("Query by checksum ->", rows)
# or by chunk_id if you stored it as primary id:
chunk_id = "c-12345"
rows2 = coll.query(expr=f'id in ["{chunk_id}"]', output_fields=["id","checksum","metadata"])
print("Query by id ->", rows2)


In [13]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings  # or fallback import

In [14]:
api_key = os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")
# many wrappers read env automatically; if yours needs explicit key, pass google_api_key=api_key
emb = GoogleGenerativeAIEmbeddings(model="models/embedding-001", google_api_key=api_key)

In [15]:
query_text = "Berapa biaya akomodasi untuk perjalanan dinas golongan 6 ke Sumatera?"
qvec = emb.embed_documents([query_text])[0]   # a single vector (list of floats)
print("Query vector dim:", len(qvec))

Query vector dim: 768


In [16]:
# coll from earlier
TOP_K = 5
# search params for HNSW/COSINE index — tune ef (higher -> higher recall, slower)
search_params = {"metric_type": "COSINE", "params": {"ef": 100}}

results = coll.search(
    data=[qvec],                 # list of query vectors (can search multiple at once)
    anns_field="embedding",      # the name of your vector field
    param=search_params,
    limit=TOP_K,
    output_fields=["id", "checksum", "metadata"]
)

# results is a list (one item per query) of "hits" lists
for hits in results:
    for hit in hits:
        # printing Hit attributes - depending on pymilvus version use .id or .entity.get("id")
        try:
            entity_id = hit.id
        except Exception:
            entity_id = hit.entity.get("id") if hasattr(hit, "entity") else None
        # score/distance: with COSINE higher is more similar (closer to 1). Many APIs expose .distance or .score
        score = getattr(hit, "score", None) or getattr(hit, "distance", None)
        metadata = hit.entity.get("metadata") if hasattr(hit, "entity") else None
        print("id:", entity_id, "score:", score, "checksum:", hit.entity.get("checksum") if hasattr(hit,"entity") else None, "meta:", metadata)


id: 32af0fdc41be44d797e14811d9180268 score: 0.8223957419395447 checksum: dc3802166461cf94c2a794d3b54723d0130deb686811136f6248fb58134aaa28 meta: {'chunk_id': '32af0fdc41be44d797e14811d9180268', 'chunk_text': 'Dalam Negeri (dalam ribuan Rupiah): Sesuai tagihan, Hotel Bintang 5 (lima) | Gol.: 10 | Luar Negeri (dalam USD): Sesuai tagihan, Hotel Bintang 5 (lima)', 'source': 'CP_Perjalanan Dinas.pdf', 'page': 27, 'element_type': 'table_row', 'table_id': 'CP_Perjalanan Dinas.pdf_p27_t0', 'row_id': 1}
id: ecd07aebc9604f7fa2336f3c5a17c055 score: 0.8050978779792786 checksum: 7a3802683712fefa08bebf71540a04e411aa5bd646005a9a9ffa009098909c55 meta: {'chunk_id': 'ecd07aebc9604f7fa2336f3c5a17c055', 'chunk_text': ": Ketentuan Lain | PERJALANAN DINAS/ BUSINESS TRAVEL: [{'': 'Ketentuan Lain', 'PERJALANAN DINAS/ BUSINESS TRAVEL': '7. Berkaitan dengan Poin 6 di atas, apabila diperlukan, CEO atau Direksi dapat mendelegasikan proses persetujuan\\npenggantianbiayaperjalananuntukpihakeksternalkepadaKaryawanden

In [17]:
for hits in results:
    for hit in hits:
        # printing Hit attributes - depending on pymilvus version use .id or .entity.get("id")
        try:
            entity_id = hit.id
        except Exception:
            entity_id = hit.entity.get("id") if hasattr(hit, "entity") else None
        # score/distance: with COSINE higher is more similar (closer to 1). Many APIs expose .distance or .score
        score = getattr(hit, "score", None) or getattr(hit, "distance", None)
        metadata = hit.entity.get("metadata") if hasattr(hit, "entity") else None
#         print("id:", entity_id, "score:", score, "checksum:", hit.entity.get("checksum") if hasattr(hit,"entity") else None, "meta:", metadata)
        print(hit.get("metadata")['chunk_text'])


Dalam Negeri (dalam ribuan Rupiah): Sesuai tagihan, Hotel Bintang 5 (lima) | Gol.: 10 | Luar Negeri (dalam USD): Sesuai tagihan, Hotel Bintang 5 (lima)
: Ketentuan Lain | PERJALANAN DINAS/ BUSINESS TRAVEL: [{'': 'Ketentuan Lain', 'PERJALANAN DINAS/ BUSINESS TRAVEL': '7. Berkaitan dengan Poin 6 di atas, apabila diperlukan, CEO atau Direksi dapat mendelegasikan proses persetujuan\npenggantianbiayaperjalananuntukpihakeksternalkepadaKaryawandengangolongan9 (sembilan)keatasdengan\nmenggunakansuratpendelegasianotoritas.\n8. Dalam hal Poin 6 di atas terjadi dan dipandang perlu melakukan pengaturan atas akomodasi penginapan dan\ntransportasi dinas Karyawan dan pihak eksternal agar berada dalam satu akomodasi dan transportasi yang sama\nuntuk mencapai tujuan dari perjalanan tersebut, maka pengaturan akomodasi penginapan dan transportasi dinas\nbagiKaryawantersebutdapatdisesuaikandenganpengaturanPerjalananDinasbagipihakeksternalyangditanggung\nolehPerusahaandengantetapmempertimbangkanefisiensiPe

In [ ]:
# Query all chunks from page 3
rows = coll.query(
    expr="page == 3", 
    output_fields=["id", "page", "metadata"]
)
print("Results:", rows)

In [18]:
all_chunks[-10]

{'chunk_id': '690857ea91054087a64cceb5a8e9d58a',
 'source': 'CP_Perjalanan Dinas.pdf',
 'page': 27,
 'element_type': 'table_row',
 'table_id': 'CP_Perjalanan Dinas.pdf_p27_t0',
 'table_index': 0,
 'row_id': 0,
 'bbox': (12.0, 60.0, 702.27, 347.762),
 'language': 'id',
 'chunk_text': 'Dalam Negeri (dalam ribuan Rupiah): Papua | Gol.: None | Luar Negeri (dalam USD): Non Asia& Afrika, Singapore, Hongkong dan Jepang (dalam USD)',
 'checksum': '5dbbfdd8fc636dc58c515917d85b6cec3688d54e40e0d6f5d9e0150c979bf0fa',
 'created_at': '2025-09-23T10:31:00Z'}

In [ ]:
for chunk in all_chunks:
    if chunk['source']=='CP_Perjalanan Dinas.pdf' and 